# Scenario 7: Adversarial Machine Learning

Welcome to this advanced scenario where we'll explore the security and robustness of our models. We'll learn how to craft **adversarial examples** to fool a neural network and discuss how to defend against such attacks.

## 1. Setup and Installations

First, we need to install the **Adversarial Robustness Toolbox (ART)** library, which provides the tools for creating and defending against attacks. We'll also need TensorFlow.

In [ ]:
!pip install adversarial-robustness-toolbox tensorflow

## 2. Import Libraries and Prepare Data

Next, we'll import all the necessary libraries and prepare our data. This process will be the same as in our previous neural network scenario for consistency.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

import art
from art.estimators.classification import KerasClassifier
from art.attacks.evasion import FastGradientMethod

# --- Data Prep ---
column_names = [
    'id', 'clump_thickness', 'unif_cell_size', 'unif_cell_shape',
    'marg_adhesion', 'single_epith_cell_size', 'bare_nuclei',
    'bland_chrom', 'norm_nucleoli', 'mitoses', 'class'
]
df = pd.read_csv('../breast-cancer-wisconsin.data', names=column_names)
df['bare_nuclei'] = df['bare_nuclei'].replace('?', np.nan)
df['bare_nuclei'] = pd.to_numeric(df['bare_nuclei'])
df.fillna(df.median(), inplace=True)
df.drop('id', axis=1, inplace=True)
df['class'] = df['class'].map({2: 0, 4: 1}) # 0 for benign, 1 for malignant

X = df.drop('class', axis=1).values
y = df['class'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 3. Train a Baseline Neural Network

We'll create and train a simple neural network, which will be the 'victim' for our attack. This is similar to the model from Scenario 5.

In [ ]:
model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0, validation_split=0.1)

# Evaluate the baseline model
y_pred_baseline = (model.predict(X_test) > 0.5).astype("int32")
accuracy_baseline = accuracy_score(y_test, y_pred_baseline)
print(f"Baseline Model Accuracy: {accuracy_baseline:.4f}")

## 4. Create an Adversarial Attack

Now for the exciting part. We will use the ART library to launch an attack against our model.

First, we need to wrap our Keras model in an `art.KerasClassifier`. This allows the ART library to interact with our model.

Then, we will use the **Fast Gradient Sign Method (FGSM)**. This attack uses the gradient of the model's loss with respect to the input data to create a new, perturbed image that maximizes the loss.

In [ ]:
# HINT: Wrap the compiled Keras model using the KerasClassifier.
# You'll need to specify the `clip_values` which represent the min and max values of your features.
# Since we scaled our data, the min/max values are not simply 0 and 1. We can use the min/max of the scaled training data as an approximation.

# YOUR CODE HERE
min_val, max_val = X_train.min(), X_train.max()
classifier = KerasClassifier(model=model, clip_values=(min_val, max_val))

print("Model wrapped in ART classifier.")

In [ ]:
# HINT: Now create an instance of the FastGradientMethod.
# You need to pass the `estimator` (our wrapped classifier) and a hyperparameter `eps`.
# `eps` controls the magnitude of the perturbation. A small `eps` makes the attack stealthier.

# YOUR CODE HERE
attack = FastGradientMethod(estimator=classifier, eps=0.1)

print("FGSM attack created.")

### Generate Adversarial Examples

With our attack object ready, we can now generate adversarial versions of our test data.

In [ ]:
# HINT: Use the `attack.generate()` method and pass in the original test data (X_test).

# YOUR CODE HERE
X_test_adv = attack.generate(x=X_test)

print("Adversarial examples generated.")

## 5. Evaluate Model Robustness

Let's see how our model performs on this new, malicious data. We expect the accuracy to drop significantly.

In [ ]:
# HINT: Use the original model to predict on the new `X_test_adv`.

# YOUR CODE HERE
y_pred_adv = (model.predict(X_test_adv) > 0.5).astype("int32")
accuracy_adv = accuracy_score(y_test, y_pred_adv)

print(f"Baseline Model Accuracy on clean data: {accuracy_baseline:.4f}")
print(f"Model Accuracy on adversarial data: {accuracy_adv:.4f}")

### What happened?

As you can see, the accuracy likely dropped dramatically! The small, carefully crafted noise was enough to fool our model.

Let's look at the difference between an original sample and its adversarial version. The changes are often very small and hard to detect by eye.

In [ ]:
sample_index = 0
original_sample = X_test[sample_index]
adv_sample = X_test_adv[sample_index]

print("Original Sample (scaled):")
print(original_sample)
print("\nAdversarial Sample (scaled):")
print(adv_sample)
print("\nDifference:")
print(adv_sample - original_sample)

## 6. Discussion and Defense

This vulnerability is a major concern for real-world systems. How could we defend against this?

One of the simplest defenses is **Adversarial Training**. The idea is to generate adversarial examples and add them to the training set. This allows the model to learn the patterns of the attack and become more robust.

**Challenge:** Can you implement adversarial training? You would need to:
1. Generate adversarial examples for your *training* data (`X_train`).
2. Combine the original training data with the new adversarial training data.
3. Retrain your model on this combined dataset.
4. Re-evaluate its performance on a *new* set of adversarial test data. Does the accuracy improve?